# v7 beta (VinBigData 512) — class160: noise audit → noise simulation → CNN

ใช้ชุดข้อมูล **`awsaf49/vinbigdata-512-image-dataset`** ดึงผ่าน `kagglehub` ลงบน Colab VM โดยตรง
แบ่งเป็น **2 คลาส**:
- **`normal` (Class 0)**: ภาพปอดปกติ (`No finding` / `class_id == 14` ไม่มีภาวะโรคอื่นปน; ~10,606 ภาพ)
- **`infiltration` (Class 1)**: ภาพปอดที่มีภาวะ `Infiltration` (`class_name == 'Infiltration'` หรือ `class_id == 6`) แม้จะมีโรคอื่นร่วมด้วยในภาพเดียวกัน (~613 ภาพ)

คัดภาพ estimated noise ต่ำตามสัดส่วนผู้สมัคร (seed 42):
- **Train**: `160 normal + 160 infiltration` (**320 ภาพ**)
- **Validation**: `40 normal + 40 infiltration` (**80 ภาพ**)
- **Test**: `50 normal + 50 infiltration` (**100 ภาพ**)

ทดลอง **12 วิธีลดและสกัดความถี่ Noise**, จำลอง noise 5 ชนิด × 3 ระดับ + ภาพก่อนเติม (รวม 16 สภาวะ), seed 42
ฝึก **1 DAE + 12 CNN (ResNet50)**; เก็บ checkpoint ทุก epoch และข้อมูลกรอบ Bounding Box สำหรับ XAI บน Google Drive

1. เลือก **Runtime → Change runtime type → GPU**
2. กด **Run all** แล้วอนุญาต mount Google Drive
3. ผลลัพธ์และข้อมูลที่คัดแล้วจะถูกบันทึกไว้ที่ `MyDrive/nih -xray/Notebook_7_beta_vinbigdata/` (หาก Colab หลุด สามารถกด Run all ใหม่เพื่อรันต่อจาก checkpoint เดิมได้ทันทีโดยไม่ต้องโหลดหรือคัดภาพใหม่)


In [1]:
#@title ติดตั้งแพ็กเกจ แตกซอร์สโค้ด และเชื่อมต่อ Google Drive
from pathlib import Path
import base64, hashlib, subprocess, sys, zipfile, io
import importlib.util
from google.colab import drive
dependencies = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "cv2": "opencv-python-headless",
    "pywt": "PyWavelets",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "nbformat": "nbformat",
    "kagglehub": "kagglehub",
}
missing = [package for module, package in dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
SOURCE_SHA256 = 'e77443bb85c769dac97abbe97e46f3c99dad65893019aefbbd1f7fa6b1e0c71e'
SOURCE_B64 = '...'
payload = base64.b64decode(SOURCE_B64)
if hashlib.sha256(payload).hexdigest() != SOURCE_SHA256:
    raise ValueError("Embedded source integrity failed")
SOURCE_STAGE = Path("/content/v7_vinbigdata_source_").with_name("v7_vinbigdata_source_" + SOURCE_SHA256[:16])
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for entry in archive.infolist():
        if not (SOURCE_STAGE / entry.filename).resolve().is_relative_to(SOURCE_STAGE.resolve()):
            raise ValueError("Source path escapes staging")
    archive.extractall(SOURCE_STAGE)
ROOT = SOURCE_STAGE / "Notebook_7_beta_vinbigdata"
loaded = sys.modules.get("v7")
if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(ROOT.resolve()):
    raise RuntimeError("Notebook source changed; restart the Colab session before loading this version")
sys.path.insert(0, str(ROOT))
from v7.colab import mount_on_fresh_path
MOUNT = mount_on_fresh_path(drive.mount)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU selected")


Mounting Google Drive for durable checkpoints and outputs...
Mounted at /content/vin_drive_v7_dnq9b22h
GPU: Tesla T4


## 1. ดาวน์โหลด `awsaf49/vinbigdata-512-image-dataset` และตั้งค่าโฟลเดอร์บน Drive

In [2]:
import os
from v7.colab import download_vinbigdata
WORK = MOUNT / "MyDrive/nih -xray/Notebook_7_beta_vinbigdata"  # โฟลเดอร์เก็บผลและ checkpoint ถาวรบน Drive
WORK.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(WORK / "cache/torch")  # reuse ImageNet weights across runtimes
DATA_ROOT = download_vinbigdata("awsaf49/vinbigdata-512-image-dataset", work=WORK)
print("VinBigData VM source:", DATA_ROOT)
print("Durable Drive outputs:", WORK)


Using Colab cache for faster access to the 'vinbigdata-512-image-dataset' dataset.
Dataset ready at: /kaggle/input/vinbigdata-512-image-dataset
Verified VinBigData source: CSV=/kaggle/input/vinbigdata-512-image-dataset/vinbigdata/train.csv, indexed images=15,000
VinBigData VM source: /kaggle/input/vinbigdata-512-image-dataset
Durable Drive outputs: /content/vin_drive_v7_dnq9b22h/MyDrive/nih -xray/Notebook_7_beta_vinbigdata


## 2. แยกคลาส `normal` vs `infiltration` → วิเคราะห์ระดับ Noise → คัดภาพเข้าชุดทดลอง
มี log สถานะทุก 10 วินาที และบันทึกชุดภาพที่คัดลง Google Drive

In [3]:
from progress import prepare_with_logs
prepare_with_logs(ROOT, DATA_ROOT, WORK, workers=4)


[v7 VinBigData audit/selection +0s] เริ่มงาน — แสดงสถานะทุก 10 วินาที; checkpoint อยู่บน Drive
[v7 VinBigData audit/selection +0s] ตรวจชุดภาพที่บันทึกไว้ / SHA-256
[v7 VinBigData audit/selection +0s] จบขั้นตอน (0.0s): ตรวจชุดภาพที่บันทึกไว้ / SHA-256
[v7 VinBigData audit/selection +0s] ตรวจ train.csv และไฟล์ภาพ VinBigData
Verified VinBigData source: CSV=/kaggle/input/vinbigdata-512-image-dataset/vinbigdata/train.csv, indexed images=15,000
[v7 VinBigData audit/selection +6s] จบขั้นตอน (6.4s): ตรวจ train.csv และไฟล์ภาพ VinBigData
[v7 VinBigData audit/selection +6s] แยกคลาส normal vs infiltration แล้ววิเคราะห์ noise
[v7 VinBigData audit/selection +10s] รอขั้นตอน: แยกคลาส normal vs infiltration แล้ววิเคราะห์ noise
[v7 VinBigData audit/selection +20s] รอขั้นตอน: แยกคลาส normal vs infiltration แล้ววิเคราะห์ noise
[v7 VinBigData audit/selection +22s] ค้นหารายชื่อไฟล์ภาพ PNG บน Colab VM (ยังไม่วิเคราะห์ noise)
[v7 VinBigData audit/selection +29s] พบไฟล์ภาพ 15,000 ไฟล์
[v7 VinBigData audit/sele

## 3. โหลดชุดภาพที่คัดแล้ว (Train 160+160, Val 40+40, Test 50+50) เข้าสู่ Colab VM

In [4]:
from v7.colab import stage_selected
LOCAL_DATA = stage_selected(WORK, Path("/content/v7_vinbigdata_cache"))
print("VM-local training images:", LOCAL_DATA)


VM-local training images: /content/v7_vinbigdata_cache/a4c34068aa3fabcb/v7_data


## 4. ฝึกสอน 1 DAE และ 12 CNN (ResNet50) — บันทึก Checkpoint ทุก Epoch ลง Drive

In [5]:
from v7.runner import run
if not torch.cuda.is_available():
    raise RuntimeError("Runtime > Change runtime type > GPU แล้ว Run all ใหม่")
EXPERIMENT = run(ROOT, LOCAL_DATA, WORK / "runs/v7_beta_vinbigdata/seed_42", device="cuda")


v7 VinBigData BETA PILOT seed 42: 12 CNN, one DAE; /content/vin_drive_v7_dnq9b22h/MyDrive/nih -xray/Notebook_7_beta_vinbigdata/runs/v7_beta_vinbigdata/seed_42
DAE 1/20: train=0.01210, validation=0.01233
DAE 2/20: train=0.00711, validation=0.00764
DAE 3/20: train=0.00533, validation=0.00505
DAE 4/20: train=0.00493, validation=0.00631
DAE 5/20: train=0.00461, validation=0.00431
DAE 6/20: train=0.00435, validation=0.00391
DAE 7/20: train=0.00417, validation=0.00395
DAE 8/20: train=0.00398, validation=0.00368
DAE 9/20: train=0.00403, validation=0.00367
DAE 10/20: train=0.00395, validation=0.00381
DAE 11/20: train=0.00394, validation=0.00389
DAE 12/20: train=0.00401, validation=0.00420
DAE 13/20: train=0.00399, validation=0.00341
DAE 14/20: train=0.00383, validation=0.00337
DAE 15/20: train=0.00364, validation=0.00351
DAE 16/20: train=0.00365, validation=0.00354
DAE 17/20: train=0.00354, validation=0.00346
DAE 18/20: train=0.00350, validation=0.00344
DAE 19/20: train=0.00372, validation=0.0

100%|██████████| 97.8M/97.8M [00:01<00:00, 102MB/s]


baseline 1/20 head_only: train=0.69069, validation=0.68061
baseline 2/20 head_only: train=0.65610, validation=0.65457
baseline 3/20 head_only: train=0.62109, validation=0.63074
baseline 4/20 layer4_head: train=0.53702, validation=0.49860
baseline 5/20 layer4_head: train=0.39519, validation=0.35235
baseline 6/20 layer4_head: train=0.26304, validation=0.24262
baseline 7/20 layer4_head: train=0.19590, validation=0.17473
baseline 8/20 layer4_head: train=0.13218, validation=0.19273
baseline 9/20 layer4_head: train=0.14404, validation=0.11768
baseline 10/20 layer4_head: train=0.10079, validation=0.11116
baseline 11/20 layer4_head: train=0.08746, validation=0.12957
baseline 12/20 layer4_head: train=0.09316, validation=0.11855
baseline 13/20 layer4_head: train=0.09223, validation=0.08667
baseline 14/20 layer4_head: train=0.06706, validation=0.11405
baseline 15/20 layer4_head: train=0.05129, validation=0.10593
baseline 16/20 layer4_head: train=0.06897, validation=0.07827
baseline 17/20 layer4_h

## 5. สรุปผลการจำแนกโรค (Normal vs Infiltration) และคุณภาพภาพ

In [6]:
import pandas as pd
REPORTS = WORK / "runs/v7_beta_vinbigdata/seed_42/reports"
table = pd.read_csv(REPORTS / "classification.csv")
display(table[["condition_id", "corruption_id", "roc_auc", "pr_auc", "balanced_accuracy", "f1", "accuracy"]])
print("Reports:", REPORTS)
print("Checkpoints / XAI loader metadata:", REPORTS.parent)


,condition_id,corruption_id,roc_auc,pr_auc,balanced_accuracy,f1,accuracy
0,baseline,clean,0.9960,0.996007,0.96,0.959184,0.96
1,baseline,gaussian_N1,0.9956,0.995672,0.94,0.937500,0.94
2,baseline,gaussian_N2,0.9976,0.997675,0.96,0.958333,0.96
3,baseline,gaussian_N3,0.9996,0.999608,0.99,0.989899,0.99
4,baseline,poisson_N1,0.9972,0.997218,0.95,0.948454,0.95
...,...,...,...,...,...,...,...
187,dft_low_mid_high_fusion,salt_pepper_N2,0.9984,0.998519,0.95,0.951456,0.95
188,dft_low_mid_high_fusion,salt_pepper_N3,0.9960,0.996457,0.94,0.942308,0.94
189,dft_low_mid_high_fusion,speckle_N1,0.9996,0.999608,0.98,0.980000,0.98
190,dft_low_mid_high_fusion,speckle_N2,0.9996,0.999608,0.98,0.980000,0.98


Reports: /content/vin_drive_v7_dnq9b22h/MyDrive/nih -xray/Notebook_7_beta_vinbigdata/runs/v7_beta_vinbigdata/seed_42/reports
Checkpoints / XAI loader metadata: /content/vin_drive_v7_dnq9b22h/MyDrive/nih -xray/Notebook_7_beta_vinbigdata/runs/v7_beta_vinbigdata/seed_42
